# Mall Customers — Customer Segmentation (K‑Means)

This notebook performs customer segmentation on the uploaded `Mall_Customers.csv` file. Steps included:

1. Load data and basic EDA
2. Preprocessing (encoding & scaling)
3. K‑Means clustering and silhouette analysis
4. PCA for 2D visualization
5. Export results

_Generated automatically by ChatGPT._

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

import matplotlib.pyplot as plt

# display settings
pd.set_option('display.max_columns', None)


In [ ]:
data_path = Path('/mnt/data/Mall_Customers.csv')
assert data_path.exists(), f'File not found: {data_path}'
df = pd.read_csv(data_path)
df.head()

In [ ]:
print('Shape:', df.shape)
print('\nInfo:')
print(df.info())

print('\nDescribe:')
print(df.describe())

## Preprocessing

We will:
- Drop or fill missing values (if any)
- Encode categorical variables (Gender)
- Scale numerical features for clustering

In [ ]:
df_clean = df.copy()
# Check for missing values
print('Missing values per column:\n', df_clean.isnull().sum())

# Simple handling (no missing values expected in typical Mall_Customers dataset)
# Encode Gender
if 'Gender' in df_clean.columns:
    df_clean['Gender'] = df_clean['Gender'].map({'Male':0, 'Female':1})

# Select features for clustering (common choice: Age, Annual Income (k$), Spending Score (1-100))
features = []
for c in ['Age','Annual Income (k$)','Spending Score (1-100)','Gender']:
    if c in df_clean.columns:
        features.append(c)
print('Using features:', features)

X = df_clean[features].copy()

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


## K‑Means clustering & silhouette score

We'll try a range of cluster counts and pick the best by silhouette score.

In [ ]:
sil_scores = {}
for k in range(2,8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, labels)
    sil_scores[k] = sil

sil_scores

best_k = max(sil_scores, key=sil_scores.get)
print(f'Best k by silhouette: {best_k} (score={sil_scores[best_k]:.4f})')

# Fit final model
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_scaled)
df_clean['Cluster'] = labels

df_clean.groupby('Cluster').mean()


## PCA projection (2D) and cluster visualization

In [ ]:
pca = PCA(n_components=2, random_state=42)
proj = pca.fit_transform(X_scaled)

plt.figure(figsize=(8,6))
for c in np.unique(labels):
    mask = labels==c
    plt.scatter(proj[mask,0], proj[mask,1], label=f'Cluster {c}')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('K-Means clusters (PCA 2D projection)')
plt.legend()
plt.grid(True)
plt.show()


## Save clustered dataset
The clustered dataset will be saved to `/mnt/data/Mall_Customers_clustered.csv`.

In [ ]:
out_path = Path('/mnt/data/Mall_Customers_clustered.csv')
df_clean.to_csv(out_path, index=False)
print('Saved clustered CSV to', out_path)

## Conclusions & Next steps

- Inspect cluster centers and segment definitions.
- Try different features, clustering algorithms (Agglomerative, DBSCAN).
- Use elbow method or silhouette alongside business context when choosing k.

---

*End of notebook.*